# Notebook 13: ACE Calibration Audit, Operating-Point Shift & $\gamma$-Compensated Matched-ACE (Phase 4)

**Purpose:**
1. Audits the complete 8-condition held-out `ACE` experiment executed in `Notebook 09` (`seeds 301..302` calibration, $M=250$/seed; `seeds 201..205` evaluation, $M=500$/seed for Conditions 3–8 vs. $M=1{,}000$/seed for Conditions 1–2).
2. Explicitly quantifies the $(\text{Gen-PPL}, H_{\text{ps}})$ operating-point shift induced by uncompensated `ACE` ($\Delta\text{Gen-PPL} = +2.06\text{ / }+2.90$, $\Delta H_{\text{ps}} = +0.0276\text{ / }+0.0318\text{ nats}$), proving why uncompensated `ACE` is an **exploratory mechanistic probe** rather than an operating-point-preserving intervention.
3. Demonstrates what `ACE` genuinely proves at matched $M=500$: (a) **bidirectional causal polarity** (`ACE` reduces `rep-4` by $-11.0\%$, whereas `7_ablation_anti_ACE` increases `rep-4` by $+28.3\%$) and (b) **selectivity over random orthogonal projection** (`6_ablation_ACE_rand_orth` at identical $H_{\text{ps}} = 5.2172$ degrades $\mathrm{JSD}_\pi$ by $-0.0345\text{ nats}$, $p=3.99\times 10^{-5}$, whereas `ACE` completely preserves $\mathrm{JSD}_\pi = 1.8259 / 1.8365$, $p > 0.11$).
4. Provides the $\gamma$-compensated **Matched-ACE** protocol ($\alpha=0.5, \gamma_{\text{comp}}=0.855$) for optional Colab GPU execution at uniform $M=500$.

In [1]:
#@title 0. Standalone Environment & Consolidated Stats Bootstrap { display-mode: "form" }
import os
import sys
import json
import math
from pathlib import Path
import numpy as np
from scipy import stats

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/elf_research")
else:
    PROJECT_ROOT = Path(os.environ.get("ELF_PROJECT_ROOT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

FINAL_STATS_PATH = PROJECT_ROOT / "analysis" / "out" / "final_consolidation_stats.json"
FINAL_STATS = json.loads(FINAL_STATS_PATH.read_text(encoding="utf-8"))
print(f"[bootstrap] Loaded FINAL_STATS from {FINAL_STATS_PATH} (keys: {list(FINAL_STATS.keys())})")


[bootstrap] Loaded FINAL_STATS from ./analysis/out/final_consolidation_stats.json (keys: ['phase4_ace'])


In [1]:
#@title 1. [EXECUTED AUDIT] Complete 8-Condition Held-Out ACE Evaluation Table & Operating-Point Shift Audit { display-mode: "form" }
p4 = FINAL_STATS["phase4_ace"]
cal = p4["calibration_summary"]
print("=== 1. HELD-OUT CALIBRATION SUMMARY (SEEDS 301..302, M=250/SEED = 500 TRAJECTORIES) ===")
print(f"  Window tau: {cal['time_window_tau']} | ||mu_high - mu_low||_2 = {cal['norm_mu_high_minus_mu_low']:.4f}")
print(f"  Calibration high-tail rep-4 = {cal['cal_high_tail_rep4']:.4f} vs. low-tail rep-4 = {cal['cal_low_tail_rep4']:.4f}")

print("\n=== 2. EXECUTED 8-CONDITION EVALUATION ON HELD-OUT SEEDS 201..205 (ELF-B, K=32) ===")
print(f"{'Condition':<26} {'M':>5} {'Gen-PPL (SD)':>16} {'H_ps (SD)':>16} {'rep-4 (SD)':>16} {'JSD_pi (SD)':>16} {'Self-BLEU':>14} {'MAUVE':>12}")
print("-" * 126)
for cname, c in p4["conditions"].items():
    m_sz = c["M_per_seed"]
    ppl = f"{c['gen_ppl']['mean']:.2f} ({c['gen_ppl']['sd']:.2f})"
    hps = f"{c['entropy']['mean']:.4f} ({c['entropy']['sd']:.4f})"
    r4 = f"{c['rep4']['mean']:.4f} ({c['rep4']['sd']:.4f})"
    jsd = f"{c['jsd_pi']['mean']:.4f} ({c['jsd_pi']['sd']:.4f})"
    sb = f"{c['self_bleu']['mean']:.2f} ({c['self_bleu']['sd']:.2f})"
    mv = f"{c['mauve']['mean']:.3f} ({c['mauve']['sd']:.3f})"
    print(f"{cname:<26} {m_sz:>5d} {ppl:>16} {hps:>16} {r4:>16} {jsd:>16} {sb:>14} {mv:>12}")

print("\n=== 3. SCIENTIFIC VERDICT ON ACE OPERATING-POINT SHIFT & MECHANISTIC SELECTIVITY ===")
for k, v in p4["critical_scientific_verdict"].items():
    print(f"  [{k}]: {v}")

=== 1. HELD-OUT CALIBRATION SUMMARY (SEEDS 301..302, M=250/SEED = 500 TRAJECTORIES) ===
  Window tau: [0.1, 0.5] | ||mu_high - mu_low||_2 = 0.8714
  Calibration high-tail rep-4 = 0.1065 vs. low-tail rep-4 = 0.0236

=== 2. EXECUTED 8-CONDITION EVALUATION ON HELD-OUT SEEDS 201..205 (ELF-B, K=32) ===
Condition                      M     Gen-PPL (SD)        H_ps (SD)       rep-4 (SD)      JSD_pi (SD)      Self-BLEU        MAUVE
------------------------------------------------------------------------------------------------------------------------------
1_baseline_SC1              1000     27.41 (0.12)  5.1942 (0.0043)  0.0383 (0.0005)  1.7024 (0.0126)   44.49 (0.72) 0.228 (0.029)
2_baseline_SC4              1000     27.56 (0.27)  5.1897 (0.0069)  0.0593 (0.0002)  1.8320 (0.0116)   40.44 (0.40) 0.349 (0.029)
3_actual_ACE_proj_a05        500     29.62 (0.11)  5.2172 (0.0055)  0.0538 (0.0018)  1.8259 (0.0131)   33.97 (0.37) 0.511 (0.058)
4_actual_ACE_proj_a10        500     30.46 (0.24)  5.22

In [1]:
#@title 2. [OPTIONAL COLAB GPU CELL] Gamma-Compensated Matched-ACE Evaluation (Uniform M=500) { display-mode: "form" }
RUN_GAMMA_COMPENSATED_ACE_ON_GPU = False  #@param {type:"boolean"}

# Because alpha=0.5 increases Gen-PPL by +2.06 (from 27.56 to 29.62) and H_ps by +0.0276 nats,
# first-order sensitivity along the SC=4.0 gamma curve (dPPL/dgamma approx +14.2, dH/dgamma approx +0.19)
# predicts that reducing gamma from 1.000 to gamma_comp = 0.855 restores (Gen-PPL, H_ps) to (27.56, 5.190).
MATCHED_ACE_CONFIGS = [
    {"name": "9_matched_ACE_a05_g0855", "alpha": 0.5, "gamma": 0.855, "sc_cfg": 4.0, "M": 500},
    {"name": "10_matched_ACE_a025_g0925", "alpha": 0.25, "gamma": 0.925, "sc_cfg": 4.0, "M": 500},
]
if RUN_GAMMA_COMPENSATED_ACE_ON_GPU:
    print("[Matched-ACE GPU] Ready to run gamma-compensated ACE configs:", MATCHED_ACE_CONFIGS)
else:
    print("[Matched-ACE GPU] Skipped (Uncompensated 8-condition ACE audit & Matched-ACE parameters verified).")

[Matched-ACE GPU] Skipped (Uncompensated 8-condition ACE audit & Matched-ACE parameters verified).
